# Homework 3 - Discontinuity at an Event and Differences-in-Differences

**Part A (Questions 1-2).** `homework_3.1.csv`, time series with an event at **time = 50**. Two
regressions are run for every series in the file:

- **Model 1 - level break only.** A jump in the value, with the slope forced to stay the same.
  $$y_t = \beta_0 + \beta_1 (t - 50) + \beta_2 D_t + \varepsilon_t, \qquad D_t = \mathbb{1}[t \ge 50]$$
  $\beta_2$ is the size of the jump at the event.
- **Model 2 - level and slope break.** The line is allowed to bend as well as jump.
  $$y_t = \beta_0 + \beta_1 (t - 50) + \beta_2 D_t + \beta_3 D_t (t - 50) + \varepsilon_t$$
  $\beta_2$ is the jump in the value, $\beta_3$ the jump in the first derivative.

Centring time on the event matters: it makes $\beta_2$ the gap between the two lines *at time 50*
rather than at time 0.

**Part B (Questions 3-5).** `homework_3.2.a.csv` and `homework_3.2.b.csv`, treatment and control
groups observed before and after. The differences-in-differences regression is
$$Y = \beta_0 + \beta_1 \text{treat} + \beta_2 \text{post} + \beta_3 (\text{treat} \times \text{post}) + \varepsilon$$
and $\beta_3$ is the treatment effect - the change in the treated group net of the change in the
control group, which is what the parallel-trends assumption buys.

| # | Question | Where |
|---|---|---|
| 1 | Strongest discontinuity in the value at t = 50 | Section 4 |
| 2 | Strongest discontinuity in the derivative at t = 50 | Section 4 |
| 3 | Which dataset has the largest treatment effect | Section 8 |
| 4 | Which has the most statistically significant effect | Section 8 |
| 5 | The treatment effect for group 2 | Section 8 |

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

CSV_ITS = "homework_3.1.csv"       # Part A
EVENT   = 50                       # the event time
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

def find_file(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"none of these exist: {candidates}")

def ols(Xmat, y, names):
    # OLS with coefficients, standard errors, t-stats, p-values and R-squared.
    Xmat = np.asarray(Xmat, float)
    y = np.asarray(y, float)
    n, k = Xmat.shape
    XtX_inv = np.linalg.inv(Xmat.T @ Xmat)
    b = XtX_inv @ Xmat.T @ y
    resid = y - Xmat @ b
    rss = resid @ resid
    tss = ((y - y.mean()) ** 2).sum()
    sigma2 = rss / (n - k)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    t = b / se
    p = 2 * stats.t.sf(np.abs(t), df=n - k)
    table = pd.DataFrame({"coefficient": b, "std_error": se, "t_stat": t, "p_value": p},
                         index=names)
    return table, {"r2": 1 - rss / tss, "resid_sd": np.sqrt(sigma2), "n": n}

# Part A - Discontinuity at the event (Questions 1-2)

## 2. Load `homework_3.1.csv`

The time column is the whole-number one; every other column is treated as a separate series to
be tested.

In [ ]:
its = pd.read_csv(CSV_ITS)

print("shape:", its.shape)
print("columns:", list(its.columns))
print("\nmissing values:\n", its.isna().sum(), sep="")
its.head()

In [ ]:
def is_whole(s):
    v = pd.to_numeric(s, errors="coerce").dropna()
    return len(v) > 0 and np.allclose(v.to_numpy(), np.round(v.to_numpy()))

TIME = next((c for c in its.columns if c.strip().lower() in {"time", "t", "period"}),
            next(c for c in its.columns if is_whole(its[c])))
SERIES = [c for c in its.columns if c != TIME]

print(f"time column -> {TIME}")
print(f"series to test -> {SERIES}")
print(f"\ntime runs {its[TIME].min()} to {its[TIME].max()}, event at {EVENT}")
print(f"observations before the event: {(its[TIME] < EVENT).sum()}, "
      f"at or after: {(its[TIME] >= EVENT).sum()}")
its.describe()

## 3. Fit both models to every series

In [ ]:
tt = its[TIME].to_numpy(float)
c  = tt - EVENT                       # time centred on the event
D  = (tt >= EVENT).astype(float)      # 1 from the event onward

model1, model2, info1, info2 = {}, {}, {}, {}
rows = []

for s in SERIES:
    y = its[s].to_numpy(float)

    # Model 1: jump in the value only, common slope.
    X1 = np.column_stack([np.ones(len(y)), c, D])
    t1, i1 = ols(X1, y, ["intercept", "slope", "jump_in_value"])

    # Model 2: jump in the value and a change of slope.
    X2 = np.column_stack([np.ones(len(y)), c, D, D * c])
    t2, i2 = ols(X2, y, ["intercept", "slope_before", "jump_in_value", "jump_in_slope"])

    model1[s], model2[s], info1[s], info2[s] = t1, t2, i1, i2

    rows.append({
        "series": s,
        "M1 jump": t1.loc["jump_in_value", "coefficient"],
        "M1 t": t1.loc["jump_in_value", "t_stat"],
        "M1 p": t1.loc["jump_in_value", "p_value"],
        "M2 jump": t2.loc["jump_in_value", "coefficient"],
        "M2 jump t": t2.loc["jump_in_value", "t_stat"],
        "M2 slope change": t2.loc["jump_in_slope", "coefficient"],
        "M2 slope t": t2.loc["jump_in_slope", "t_stat"],
        "M2 slope p": t2.loc["jump_in_slope", "p_value"],
        "slope before": t2.loc["slope_before", "coefficient"],
        "resid_sd": i2["resid_sd"],
    })

its_summary = pd.DataFrame(rows).set_index("series")
its_summary

In [ ]:
for s in SERIES:
    print(f"===== series {s} =====")
    print("Model 1 - jump in the value only:")
    print(model1[s])
    print(f"  R^2 = {info1[s]['r2']:.6f}")
    print("\nModel 2 - jump in the value and in the slope:")
    print(model2[s])
    print(f"  R^2 = {info2[s]['r2']:.6f}\n")

## 4. Questions 1-2 - which series breaks hardest

Two rankings are printed. The **t-statistic** ranking is the one to trust when the series sit on
different scales, since a large jump in a noisy series is weaker evidence than a small jump in a
clean one. The raw-size ranking is there in case the question means the biggest number.

Question 1 reads off Model 1 (value only), matching the instruction to consider that case first.
Question 2 reads the interaction term in Model 2.

In [ ]:
q1_by_t    = its_summary["M1 t"].abs().idxmax()
q1_by_size = its_summary["M1 jump"].abs().idxmax()
q2_by_t    = its_summary["M2 slope t"].abs().idxmax()
q2_by_size = its_summary["M2 slope change"].abs().idxmax()

print("Q1  discontinuity in the VALUE at the event (Model 1)")
for s in SERIES:
    print(f"      {s}: jump {its_summary.loc[s, 'M1 jump']:+10.6f}   "
          f"t {its_summary.loc[s, 'M1 t']:8.3f}   p {its_summary.loc[s, 'M1 p']:.3g}")
print(f"      -> by t-statistic: {q1_by_t}    by raw size: {q1_by_size}\n")

print("Q2  discontinuity in the DERIVATIVE at the event (Model 2 interaction)")
for s in SERIES:
    print(f"      {s}: slope change {its_summary.loc[s, 'M2 slope change']:+10.6f}   "
          f"t {its_summary.loc[s, 'M2 slope t']:8.3f}   p {its_summary.loc[s, 'M2 slope p']:.3g}")
print(f"      -> by t-statistic: {q2_by_t}    by raw size: {q2_by_size}")

In [ ]:
# One panel per series: the data with the fitted broken line from Model 2.
ncol = min(2, len(SERIES))
nrow = int(np.ceil(len(SERIES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 4.2 * nrow), squeeze=False)

for ax, s in zip(axes.ravel(), SERIES):
    y = its[s].to_numpy(float)
    ax.scatter(tt, y, s=14, alpha=0.55, color="grey")
    b = model2[s]["coefficient"]
    lo = tt[tt < EVENT]
    hi = tt[tt >= EVENT]
    ax.plot(lo, b["intercept"] + b["slope_before"] * (lo - EVENT), color="steelblue", lw=2.5,
            label=f"before: slope {b['slope_before']:.4f}")
    ax.plot(hi, b["intercept"] + b["jump_in_value"]
            + (b["slope_before"] + b["jump_in_slope"]) * (hi - EVENT),
            color="crimson", lw=2.5,
            label=f"after: slope {b['slope_before'] + b['jump_in_slope']:.4f}")
    ax.axvline(EVENT, color="black", ls="--", lw=1.2)
    ax.set_title(f"{s}  -  jump {b['jump_in_value']:+.3f} (t {model2[s].loc['jump_in_value', 't_stat']:.2f}), "
                 f"slope change {b['jump_in_slope']:+.3f} (t {model2[s].loc['jump_in_slope', 't_stat']:.2f})",
                 fontsize=9)
    ax.set_xlabel(TIME); ax.set_ylabel(s)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

for ax in axes.ravel()[len(SERIES):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Part B - Differences-in-differences (Questions 3-5)

## 5. Load the two datasets

The treatment indicator and the before/after indicator are found by name where possible, and
text labels such as `treated` / `control` or `before` / `after` are mapped to 1 / 0.

In [ ]:
path_a = find_file("homework_3.2.a.csv", "homework_3.2.a", "homework_3.2a.csv")
path_b = find_file("homework_3.2.b.csv", "homework_3.2.b", "homework_3.2b.csv")
print("dataset a:", path_a)
print("dataset b:", path_b)

TREAT_WORDS = ("treat", "group", "grp", "treated", "d")
POST_WORDS  = ("post", "after", "time", "period", "t")

def to_binary(s: pd.Series):
    # Map a column to 0/1 whether it is already numeric or holds text labels.
    if pd.api.types.is_numeric_dtype(s):
        vals = sorted(s.dropna().unique())
        if len(vals) == 2:
            return (s == vals[1]).astype(float), {vals[0]: 0, vals[1]: 1}
        return None, None
    low = s.astype(str).str.strip().str.lower()
    one = low.str.contains("treat|after|post|1|yes|true", regex=True)
    zero = low.str.contains("control|before|pre|0|no|false", regex=True)
    if (one | zero).all():
        return one.astype(float), "text labels mapped"
    return None, None

def detect_cols(df):
    cols = list(df.columns)
    treat_col = next((c for c in cols if any(w in c.strip().lower() for w in TREAT_WORDS)), None)
    post_col  = next((c for c in cols if c != treat_col
                      and any(w in c.strip().lower() for w in POST_WORDS)), None)
    out_col   = next((c for c in cols if c.strip().upper() in {"Y", "OUTCOME", "VALUE"}), None)

    remaining = [c for c in cols if c not in {treat_col, post_col, out_col}]
    if out_col is None:
        out_col = max(remaining, key=lambda c: df[c].nunique())
        remaining.remove(out_col)
    if treat_col is None:
        treat_col = remaining.pop(0)
    if post_col is None:
        post_col = remaining.pop(0)
    return treat_col, post_col, out_col

def tidy_did(df, treat_col, post_col, out_col):
    treat, _ = to_binary(df[treat_col])
    post, _  = to_binary(df[post_col])
    if treat is None or post is None:
        raise ValueError(f"could not turn {treat_col}/{post_col} into 0-1 indicators")
    return pd.DataFrame({"treat": treat, "post": post,
                         "y": pd.to_numeric(df[out_col], errors="coerce")}).dropna().reset_index(drop=True)

def load_did(path, label):
    df = pd.read_csv(path)
    treat_col, post_col, out_col = detect_cols(df)
    out = tidy_did(df, treat_col, post_col, out_col)
    print(f"dataset {label}: treatment -> '{treat_col}', before/after -> '{post_col}', "
          f"outcome -> '{out_col}', n = {len(out)}")
    return out

did = {"a": load_did(path_a, "a"), "b": load_did(path_b, "b")}
did["a"].head()

In [ ]:
# The four group means. The effect is the change in the treated group minus the change
# in the control group.
for k, df in did.items():
    cell = df.pivot_table(index="treat", columns="post", values="y", aggfunc="mean")
    cell.index = ["control", "treated"]
    cell.columns = ["before", "after"]
    cell["change"] = cell["after"] - cell["before"]
    print(f"===== dataset {k} =====")
    print(cell)
    print(f"difference-in-differences from the means: "
          f"{cell.loc['treated', 'change'] - cell.loc['control', 'change']:.6f}")
    print(f"counts per cell:\n{df.groupby(['treat', 'post']).size()}\n")

## 6. The regression

In [ ]:
did_tables = {}
rows = []
for k, df in did.items():
    tr = df["treat"].to_numpy(float)
    po = df["post"].to_numpy(float)
    X = np.column_stack([np.ones(len(df)), tr, po, tr * po])
    tab, inf = ols(X, df["y"].to_numpy(float),
                   ["intercept", "treat", "post", "treat_x_post"])
    did_tables[k] = tab
    rows.append({
        "dataset": k,
        "effect": tab.loc["treat_x_post", "coefficient"],
        "std_error": tab.loc["treat_x_post", "std_error"],
        "t_stat": tab.loc["treat_x_post", "t_stat"],
        "p_value": tab.loc["treat_x_post", "p_value"],
        "n": inf["n"],
        "resid_sd": inf["resid_sd"],
    })
    print(f"===== dataset {k} =====")
    print(tab)
    print(f"  R^2 = {inf['r2']:.6f},  n = {inf['n']}\n")

did_summary = pd.DataFrame(rows).set_index("dataset")
did_summary["abs_t"] = did_summary["t_stat"].abs()
did_summary

## 7. The picture

Each dataset gets its two-line plot: the control group's before-and-after path, and the treated
group's. Under parallel trends the dashed line is where the treated group would have landed
without the treatment, and the gap to the solid point is the effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, k in zip(axes, did):
    df = did[k]
    cell = df.pivot_table(index="treat", columns="post", values="y", aggfunc="mean")
    x = [0, 1]
    ctrl = [cell.loc[0, 0], cell.loc[0, 1]]
    trt  = [cell.loc[1, 0], cell.loc[1, 1]]
    counter = trt[0] + (ctrl[1] - ctrl[0])

    ax.plot(x, ctrl, "o-", color="steelblue", lw=2, label="control")
    ax.plot(x, trt, "o-", color="crimson", lw=2, label="treated")
    ax.plot(x, [trt[0], counter], "o--", color="grey", lw=2, label="treated, parallel trend")
    ax.annotate("", xy=(1, trt[1]), xytext=(1, counter),
                arrowprops=dict(arrowstyle="<->", color="black", lw=1.5))
    ax.text(1.02, (trt[1] + counter) / 2, f"effect {trt[1] - counter:+.3f}", fontsize=9)

    ax.set_xticks(x); ax.set_xticklabels(["before", "after"])
    ax.set_xlim(-0.2, 1.4)
    ax.set_ylabel("mean outcome")
    ax.set_title(f"dataset {k}  -  effect {did_summary.loc[k, 'effect']:+.4f} "
                 f"(t {did_summary.loc[k, 't_stat']:.2f})")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Questions 3-5

- **Q3** - the largest treatment effect is the biggest interaction coefficient.
- **Q4** - the most significant is the largest $|t|$, which accounts for the standard error; a big
  effect measured noisily can be less convincing than a smaller, cleanly measured one.
- **Q5** - "group 2" is read as the second dataset, `homework_3.2.b.csv`. If your file instead has
  a column of group labels with a value `2` in it, the cell after this one estimates the effect
  separately for each label.

In [ ]:
largest = did_summary["effect"].abs().idxmax()
most_sig = did_summary["abs_t"].idxmax()

print("Q3  treatment effect by dataset")
for k in did:
    print(f"      dataset {k}: effect {did_summary.loc[k, 'effect']:+.6f} "
          f"(se {did_summary.loc[k, 'std_error']:.6f})")
print(f"      -> largest: dataset {largest}\n")

print("Q4  significance of each effect")
for k in did:
    print(f"      dataset {k}: t {did_summary.loc[k, 't_stat']:8.3f}   "
          f"p {did_summary.loc[k, 'p_value']:.3g}")
print(f"      -> most significant: dataset {most_sig}\n")

print("Q5  treatment effect for group 2 (reading group 2 as dataset b)")
print(f"      {did_summary.loc['b', 'effect']:.6f}   "
      f"(se {did_summary.loc['b', 'std_error']:.6f}, "
      f"t {did_summary.loc['b', 't_stat']:.3f})")
print(f"      dataset a for comparison: {did_summary.loc['a', 'effect']:.6f}")

In [ ]:
def did_effect(frame):
    tr = frame["treat"].to_numpy(float)
    po = frame["post"].to_numpy(float)
    X = np.column_stack([np.ones(len(frame)), tr, po, tr * po])
    tab, _ = ols(X, frame["y"].to_numpy(float),
                 ["intercept", "treat", "post", "treat_x_post"])
    return tab.loc["treat_x_post"]

# Only relevant if a file carries an explicit group label with more than two values -
# then each label gets its own differences-in-differences estimate.
found_any = False
for path, label in [(path_a, "a"), (path_b, "b")]:
    raw = pd.read_csv(path)
    gcols = [c for c in raw.columns
             if any(w in c.strip().lower() for w in ("group", "grp")) and raw[c].nunique() > 2]
    for gc in gcols:
        found_any = True
        treat_col, post_col, out_col = detect_cols(raw.drop(columns=[gc]))
        print(f"dataset {label}, per-group estimates from column '{gc}':")
        for lab, sub in raw.groupby(gc):
            try:
                res = did_effect(tidy_did(sub, treat_col, post_col, out_col))
                print(f"    group {lab}: effect {res['coefficient']:+.6f}   "
                      f"se {res['std_error']:.6f}   t {res['t_stat']:.3f}")
            except Exception as e:
                print(f"    group {lab}: could not estimate ({e})")

if not found_any:
    print("no multi-valued group column in either file, "
          "so group 2 is read as dataset b (homework_3.2.b.csv)")

## 9. All answers

In [ ]:
print("PART A  (homework_3.1.csv, event at time 50)")
print(f"  Q1  strongest discontinuity in the value    -> {q1_by_t}  "
      f"(jump {its_summary.loc[q1_by_t, 'M1 jump']:+.6f}, t {its_summary.loc[q1_by_t, 'M1 t']:.3f})")
print(f"  Q2  strongest discontinuity in the slope    -> {q2_by_t}  "
      f"(change {its_summary.loc[q2_by_t, 'M2 slope change']:+.6f}, "
      f"t {its_summary.loc[q2_by_t, 'M2 slope t']:.3f})\n")

print("PART B  (homework_3.2.a / homework_3.2.b)")
print(f"  Q3  largest treatment effect                -> dataset {largest} "
      f"({did_summary.loc[largest, 'effect']:+.6f})")
print(f"  Q4  most statistically significant          -> dataset {most_sig} "
      f"(t {did_summary.loc[most_sig, 't_stat']:.3f})")
print(f"  Q5  treatment effect for group 2 (dataset b): "
      f"{did_summary.loc['b', 'effect']:.6f}")